# Phase 2 Emotion System Validation

This notebook checks the Phase 2 emotion system with deterministic, local tests:

1. High-valence misinformation should share more readily than low-valence content.
2. Higher arousal should increase sharing probability.
3. Higher arousal should narrow HK tolerance and change updates.

In [1]:
from pathlib import Path
import sys

import numpy as np

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from backend.sim.agent import HKAgent, StubbornAgent, initialize_agents
from backend.sim.content import Content
from backend.sim.simulation import _process_agent_tick

print(f'Project root: {PROJECT_ROOT}')
print('Imports ready')

Project root: /home/chicocaine/projects/modeling-simulation/echo-chamber-sim
Imports ready


In [2]:
def make_content(valence: float, misinfo_score: float, content_id: int = 1) -> Content:
    return Content(
        id=content_id,
        creator_id=0,
        timestamp=0,
        ideological_score=0.1,
        emotional_valence=valence,
        misinfo_score=misinfo_score,
        virality=0.5,
        source_credibility=0.5,
        is_misinformation=bool(misinfo_score > 0.5),
        belief_update_weight=0.5,
        topic_vector=[],
        coordinated_campaign_id=None,
        is_satire=False,
    )

def share_rate_for_content(agent, content, trials=500, seed=123):
    shared = 0
    for trial in range(trials):
        _, _, _, items = _process_agent_tick(
            agent=agent,
            feed=[content],
            neighbors=[],
            weights={},
            alpha=0.65,
            emotional_decay=0.85,
            arousal_share_weight=0.3,
            valence_share_weight=0.4,
            random_seed=seed + trial,
        )
        shared += len(items)
    return shared / trials

def make_stubborn_agent(agent_id=1, arousal=0.2):
    return StubbornAgent(
        id=agent_id,
        agent_type='stubborn',
        opinion=0.0,
        initial_opinion=0.0,
        stubbornness=0.2,
        susceptibility=0.5,
        trust=0.5,
        expertise=0.5,
        activity_rate=0.5,
        emotional_arousal=arousal,
        media_literacy=0.5,
        confidence_bound=0.3,
        arousal_tolerance_effect=0.4,
        contrarian_prob=0.0,
        influence_weight_multiplier=1.0,
        suspicion_score=0.0,
        is_active=True,
        sir_state='S',
        opinion_history=[0.0],
        misinfo_rate=0.0,
    )

print('Helpers ready')

Helpers ready


In [3]:
# 1) High-valence misinformation should share more readily than low-valence content.
rng = np.random.default_rng(7)
high_valence_misinfo = make_content(valence=0.9, misinfo_score=0.9, content_id=1)
low_valence_content = make_content(valence=0.1, misinfo_score=0.1, content_id=2)
base_agent = make_stubborn_agent(arousal=0.2)

high_rate = share_rate_for_content(base_agent, high_valence_misinfo, trials=500, seed=100)
low_rate = share_rate_for_content(base_agent, low_valence_content, trials=500, seed=200)
print(f'High-valence misinformation share rate: {high_rate:.3f}')
print(f'Low-valence content share rate:        {low_rate:.3f}')
print(f'Delta: {high_rate - low_rate:.3f}')
assert high_rate > low_rate

# 2) Higher arousal should increase sharing probability.
low_arousal_agent = make_stubborn_agent(agent_id=2, arousal=0.0)
high_arousal_agent = make_stubborn_agent(agent_id=3, arousal=1.0)
fixed_content = make_content(valence=0.7, misinfo_score=0.8, content_id=3)

low_arousal_rate = share_rate_for_content(low_arousal_agent, fixed_content, trials=500, seed=300)
high_arousal_rate = share_rate_for_content(high_arousal_agent, fixed_content, trials=500, seed=400)
print(f'Low-arousal share rate:  {low_arousal_rate:.3f}')
print(f'High-arousal share rate: {high_arousal_rate:.3f}')
print(f'Delta: {high_arousal_rate - low_arousal_rate:.3f}')
assert high_arousal_rate > low_arousal_rate

# 3) Higher arousal should narrow HK tolerance and change the update.
neighbors = [
    StubbornAgent(
        id=10,
        agent_type='stubborn',
        opinion=0.24,
        initial_opinion=0.24,
        stubbornness=1.0,
        susceptibility=0.5,
        trust=0.5,
        expertise=0.5,
        activity_rate=0.5,
        emotional_arousal=0.0,
        media_literacy=0.5,
        confidence_bound=0.3,
        arousal_tolerance_effect=0.4,
        contrarian_prob=0.0,
        influence_weight_multiplier=1.0,
        suspicion_score=0.0,
        is_active=True,
        sir_state='S',
        opinion_history=[0.24],
        misinfo_rate=0.0,
    ),
    StubbornAgent(
        id=11,
        agent_type='stubborn',
        opinion=0.15,
        initial_opinion=0.15,
        stubbornness=1.0,
        susceptibility=0.5,
        trust=0.5,
        expertise=0.5,
        activity_rate=0.5,
        emotional_arousal=0.0,
        media_literacy=0.5,
        confidence_bound=0.3,
        arousal_tolerance_effect=0.4,
        contrarian_prob=0.0,
        influence_weight_multiplier=1.0,
        suspicion_score=0.0,
        is_active=True,
        sir_state='S',
        opinion_history=[0.15],
        misinfo_rate=0.0,
    ),
]

hk_low = HKAgent(
    id=20,
    agent_type='hk',
    opinion=0.0,
    initial_opinion=0.0,
    stubbornness=0.0,
    susceptibility=0.5,
    trust=0.5,
    expertise=0.5,
    activity_rate=0.5,
    emotional_arousal=0.0,
    media_literacy=0.5,
    confidence_bound=0.3,
    arousal_tolerance_effect=0.4,
    contrarian_prob=0.0,
    influence_weight_multiplier=1.0,
    suspicion_score=0.0,
    is_active=True,
    sir_state='S',
    opinion_history=[0.0],
    misinfo_rate=0.0,
)
hk_high = HKAgent(
    id=21,
    agent_type='hk',
    opinion=0.0,
    initial_opinion=0.0,
    stubbornness=0.0,
    susceptibility=0.5,
    trust=0.5,
    expertise=0.5,
    activity_rate=0.5,
    emotional_arousal=1.0,
    media_literacy=0.5,
    confidence_bound=0.3,
    arousal_tolerance_effect=0.4,
    contrarian_prob=0.0,
    influence_weight_multiplier=1.0,
    suspicion_score=0.0,
    is_active=True,
    sir_state='S',
    opinion_history=[0.0],
    misinfo_rate=0.0,
)

hk_low_update = hk_low.compute_update(neighbors, {10: 1.0, 11: 1.0})
hk_high_update = hk_high.compute_update(neighbors, {10: 1.0, 11: 1.0})
print(f'HK low-arousal update:  {hk_low_update:.3f}')
print(f'HK high-arousal update: {hk_high_update:.3f}')
assert hk_low_update != hk_high_update

print('Phase 2 emotion system validation checks passed.')

High-valence misinformation share rate: 0.284
Low-valence content share rate:        0.228
Delta: 0.056
Low-arousal share rate:  0.272
High-arousal share rate: 0.306
Delta: 0.034
HK low-arousal update:  0.195
HK high-arousal update: 0.150
Phase 2 emotion system validation checks passed.
